# carGO PH — Blockchain Data Retrieval
**Course:** MO-IT148 — Application Development and Emerging Technologies  
**Week 6:** Data Retrieval + Cleaning + Stats  
**Group:** NodeBlk

Retrieves IoT sensor records (GPS, Temperature, RFID) from the IoTDataStorage smart contract on Ganache via Web3.py. Cleans and structures the data into a unified DataFrame for statistical analysis and export.

> ⚠️ Prerequisite: Ganache must be running with the contract deployed before executing the Setup cell.

## 1. Setup
Imports, ABI load, and contract instantiation. Ganache must be running before this cell executes. All downstream cells depend on `logistics_contract`.

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Imports, ABI load, contract instantiation, connection confirmation

from web3 import Web3
import pandas as pd
import numpy as np
import json

# Load ABI from compiled contract data
with open('../contracts/IoTDataStorage_compData.json') as f:
    comp_data = json.load(f)

abi = json.loads(comp_data['metadata'])['output']['abi']
# comp_data['metadata'] is a JSON string inside the JSON file — needs a second parse

# Contract address — update this each new Ganache session
CONTRACT_ADDRESS = Web3.to_checksum_address("0x1A0F7425bf9Ba7773e00266486AAAB5b2ebE7e5a")

# Connect to Ganache
web3 = Web3(Web3.HTTPProvider("http://127.0.0.1:7545"))
logistics_contract = web3.eth.contract(address=CONTRACT_ADDRESS, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

# Confirm connection
print("Connected:", web3.is_connected())
print("GPS records on chain:", logistics_contract.functions.gpsRecordCount().call())
print("Temp records on chain:", logistics_contract.functions.tempRecordCount().call())
print("RFID records on chain:", logistics_contract.functions.rfidRecordCount().call())

Connected: True
GPS records on chain: 150
Temp records on chain: 95
RFID records on chain: 84


## 2. Retrieval
Loops through GPS, Temperature, and RFID records stored on-chain. Collects raw records into lists for DataFrame construction.

In [2]:
# ── Retrieval ───────────────────────────────────────────────────────────────

# GPS records
gps_count = logistics_contract.functions.gpsRecordCount().call()
gps_rows = []
for i in range(gps_count):
    r = logistics_contract.functions.gpsRecords(i).call()
    gps_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "latitude": r[3], "longitude": r[4], "sensor_type": "GPS"
    })
gps_records_df = pd.DataFrame(gps_rows)
if gps_records_df.empty:
    print("⚠️ Warning: No GPS records retrieved. Check Ganache session.")

# Temperature records
temp_count = logistics_contract.functions.tempRecordCount().call()
temp_rows = []
for i in range(temp_count):
    r = logistics_contract.functions.tempRecords(i).call()
    temp_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "temperature_raw": r[3], "sensor_type": "Temperature"
    })
temp_records_df = pd.DataFrame(temp_rows)
if temp_records_df.empty:
    print("⚠️ Warning: No Temperature records retrieved. Check Ganache session.")

# RFID records
rfid_count = logistics_contract.functions.rfidRecordCount().call()
rfid_rows = []
for i in range(rfid_count):
    r = logistics_contract.functions.rfidRecords(i).call()
    rfid_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "scan_status": r[3], "sensor_type": "RFID"
    })
rfid_records_df = pd.DataFrame(rfid_rows)
if rfid_records_df.empty:
    print("⚠️ Warning: No RFID records retrieved. Check Ganache session.")

## 3. DataFrame Construction
Converts raw record lists into structured DataFrames — one per sensor type.

In [3]:
# ── DataFrame Construction ──────────────────────────────────────────────────

iot_records_df = pd.concat(
    [gps_records_df, temp_records_df, rfid_records_df],
    ignore_index=True
)
# ignore_index=True — resets to clean 0-based index after stacking three DataFrames
# columns absent for a sensor type (e.g. latitude for Temperature rows) become NaN

print(f"Total records retrieved: {len(iot_records_df)}")
print(iot_records_df.dtypes)

Total records retrieved: 329
timestamp            int64
rfid_tag               str
device_id              str
latitude               str
longitude              str
sensor_type            str
temperature_raw    float64
scan_status            str
dtype: object


## 4. Cleaning
Type conversions and data quality. Unix timestamps → datetime, tempTimes10 ÷ 10 → °C, drop NaN, reset index.

In [7]:
# ── Cleaning ────────────────────────────────────────────────────────────────

iot_cleaned_df = iot_records_df.copy()
# copy() — keeps iot_records_df intact; avoids SettingWithCopyWarning

# Timestamp — Unix int → datetime
iot_cleaned_df['timestamp'] = pd.to_datetime(iot_cleaned_df['timestamp'], unit='s')
# unit='s' — integers are Unix seconds since 1970-01-01

# Latitude/longitude — string → float (stored as strings in Solidity contract)
iot_cleaned_df['latitude'] = pd.to_numeric(iot_cleaned_df['latitude'], errors='coerce')
iot_cleaned_df['longitude'] = pd.to_numeric(iot_cleaned_df['longitude'], errors='coerce')

# Temperature — int × 10 → float °C
iot_cleaned_df['temperature_c'] = iot_cleaned_df['temperature_raw'] / 10
iot_cleaned_df.drop(columns=['temperature_raw'], inplace=True)
# drop raw column after conversion — no redundant data in cleaned DataFrame

# Drop NaN on identity fields only — type-specific columns (latitude, temperature_c, scan_status)
# are expected to be NaN for non-matching sensor types
iot_cleaned_df.dropna(subset=['rfid_tag', 'sensor_type'], inplace=True)

# Reset index after drops
iot_cleaned_df.reset_index(drop=True, inplace=True)

print(f"Cleaned records: {len(iot_cleaned_df)}")
print(iot_cleaned_df.dtypes)
print(iot_cleaned_df.head())

Cleaned records: 329
timestamp        datetime64[s]
rfid_tag                   str
device_id                  str
latitude               float64
longitude              float64
sensor_type                str
scan_status                str
temperature_c          float64
dtype: object
            timestamp  rfid_tag device_id   latitude   longitude sensor_type  \
0 2026-06-03 09:54:46  RFID-020    GPS291  14.601956  120.989036         GPS   
1 2026-06-03 09:54:46  RFID-011    GPS728  14.620032  120.962165         GPS   
2 2026-06-03 09:54:48  RFID-020    GPS920  14.595757  120.981906         GPS   
3 2026-06-03 09:54:48  RFID-028    GPS356  14.374937  121.038127         GPS   
4 2026-06-03 09:54:50  RFID-011    GPS262  14.622406  120.970944         GPS   

  scan_status  temperature_c  
0         NaN            NaN  
1         NaN            NaN  
2         NaN            NaN  
3         NaN            NaN  
4         NaN            NaN  


## 5. RFID Filter
Filters and groups records by rfid_tag for per-shipment analysis.

In [8]:
# ── RFID Filter ─────────────────────────────────────────────────────────────

iot_by_rfid = iot_cleaned_df.groupby('rfid_tag')
# groupby returns a GroupBy object — not a DataFrame
# use .get_group("RFID-001") in Stats to access one shipment's records

print(f"Unique shipments (RFID tags): {iot_by_rfid.ngroups}")
print("Groups:", list(iot_by_rfid.groups.keys()))

Unique shipments (RFID tags): 30
Groups: ['RFID-001', 'RFID-002', 'RFID-003', 'RFID-004', 'RFID-005', 'RFID-006', 'RFID-007', 'RFID-008', 'RFID-009', 'RFID-010', 'RFID-011', 'RFID-012', 'RFID-013', 'RFID-014', 'RFID-015', 'RFID-016', 'RFID-017', 'RFID-018', 'RFID-019', 'RFID-020', 'RFID-021', 'RFID-022', 'RFID-023', 'RFID-024', 'RFID-025', 'RFID-026', 'RFID-027', 'RFID-028', 'RFID-029', 'RFID-030']


## 6. Stats
NumPy descriptive statistics per sensor type — mean, min, max, std.

In [12]:
# ── Stats ───────────────────────────────────────────────────────────────────

stats_rows = []
for sensor_type, group in iot_cleaned_df.groupby('sensor_type'):
    if sensor_type == 'Temperature':
        vals = group['temperature_c'].dropna()
        col = 'temperature_c'
    elif sensor_type == 'GPS':
        vals = group['latitude'].dropna()
        col = 'latitude'
    else:
        continue  # RFID has no numeric column — skip

    if not vals.empty:
        stats_rows.append({
            'sensor_type': sensor_type,
            'column': col,
            'mean': np.mean(vals),
            'min': np.min(vals),
            'max': np.max(vals),
            'std': np.std(vals)
        })

sensor_stats_df = pd.DataFrame(stats_rows)
print(sensor_stats_df)

   sensor_type         column       mean        min        max        std
0          GPS       latitude  14.578048  14.264923  17.962386   0.395666
1  Temperature  temperature_c  -2.561053 -29.900000  16.400000  16.111769


## 7. Export
Pivot table construction and CSV export for downstream visualization.

In [13]:
# ── Export ──────────────────────────────────────────────────────────────────

# Full cleaned dataset — primary Tableau source
# Tidy format: one row per record, one column per variable
iot_cleaned_df.to_csv('../data/iot_cleaned_data.csv', index=False)
print("iot_cleaned_data.csv exported.")

# NumPy stats summary — sensor-level summary for dashboard cards
# Already tidy from Stats cell — sensor_type, column, mean, min, max, std
sensor_stats_df.to_csv('../data/sensor_stats_summary.csv', index=False)
print("sensor_stats_summary.csv exported.")

iot_cleaned_data.csv exported.
sensor_stats_summary.csv exported.


## 8. Preview
Sample output rows from each DataFrame.

In [14]:
# ── Preview ─────────────────────────────────────────────────────────────────

print("Pipeline complete. Sample records from iot_cleaned_df:")
display(iot_cleaned_df.head(10))

print(f"\nTotal clean records: {len(iot_cleaned_df)}")
print(f"Unique shipments: {iot_cleaned_df['rfid_tag'].nunique()}")
print(f"Sensor types: {iot_cleaned_df['sensor_type'].unique()}")

Pipeline complete. Sample records from iot_cleaned_df:


,timestamp,rfid_tag,device_id,latitude,longitude,sensor_type,scan_status,temperature_c
0,2026-06-03 09:54:46,RFID-020,GPS291,14.601956,120.989036,GPS,NaN,NaN
1,2026-06-03 09:54:46,RFID-011,GPS728,14.620032,120.962165,GPS,NaN,NaN
2,2026-06-03 09:54:48,RFID-020,GPS920,14.595757,120.981906,GPS,NaN,NaN
3,2026-06-03 09:54:48,RFID-028,GPS356,14.374937,121.038127,GPS,NaN,NaN
4,2026-06-03 09:54:50,RFID-011,GPS262,14.622406,120.970944,GPS,NaN,NaN
5,2026-06-03 09:54:52,RFID-020,GPS343,14.544307,120.958798,GPS,NaN,NaN
6,2026-06-03 09:54:53,RFID-028,GPS501,14.474628,121.042294,GPS,NaN,NaN
7,2026-06-03 09:54:53,RFID-011,GPS125,14.552821,120.960382,GPS,NaN,NaN
8,2026-06-03 09:54:55,RFID-020,GPS858,14.550873,120.954035,GPS,NaN,NaN
9,2026-06-03 09:54:56,RFID-028,GPS805,14.521048,121.099622,GPS,NaN,NaN



Total clean records: 329
Unique shipments: 30
Sensor types: <StringArray>
['GPS', 'Temperature', 'RFID']
Length: 3, dtype: str
